# 🤖 Loan Prediction — Phase 2: Modelling & Predictions
**Pipeline:** Load preprocessed data → Train models → Compare → Predict new customers

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Binarizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve)
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek
from xgboost import XGBClassifier
import joblib, warnings
warnings.filterwarnings('ignore')
sns.set_style("whitegrid")


## 1. Load Preprocessed Data (from Phase 1)

In [ ]:
df = pd.read_csv('preprocessed_data.csv')

X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print("\nClass distribution (train):")
print(y_train.value_counts())
print(f"\nImbalance ratio: {y_train.value_counts()[1]/y_train.value_counts()[0]:.2f}:1")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = y_train.value_counts()
axes[0].bar(['Rejected (0)', 'Approved (1)'], counts.values,
            color=['#FF5252', '#4CAF50'])
axes[0].set_title('Class Distribution — Training Set', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 3, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=['Rejected', 'Approved'],
            autopct='%1.1f%%', colors=['#FF5252','#4CAF50'], startangle=90)
axes[1].set_title('Class Proportion — Training Set', fontweight='bold')

plt.tight_layout()
plt.savefig('08_class_distribution.png', dpi=150)
plt.show()


In [ ]:
def plot_confusion_matrix(ax, y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Rejected','Approved'],
                yticklabels=['Rejected','Approved'])
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

def evaluate(name, y_true, y_pred, y_proba=None):
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    if y_proba is not None:
        print(f"  ROC-AUC  : {roc_auc_score(y_true, y_proba):.4f}")
    print("\n  Classification Report:")
    print(classification_report(y_true, y_pred))


## 2. XGBoost Classifier

In [ ]:
xgb_params = dict(max_depth=3, learning_rate=0.02, n_estimators=2000,
                  subsample=0.7, colsample_bytree=0.7,
                  min_child_weight=1, gamma=0.1, reg_alpha=3, reg_lambda=3,
                  early_stopping_rounds=50, random_state=42)

# Baseline
xgb = XGBClassifier(**xgb_params)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
evaluate("XGBoost Baseline", y_test, xgb.predict(X_test),
         xgb.predict_proba(X_test)[:,1])


In [ ]:
# Strategy 1 — SMOTE
smote = SMOTE(random_state=42)
X_sm, y_sm = smote.fit_resample(X_train, y_train)

# Strategy 4 — ADASYN
adasyn = ADASYN(random_state=42)
X_ad, y_ad = adasyn.fit_resample(X_train, y_train)

# Strategy 5 — SMOTE-Tomek
X_st, y_st = SMOTETomek(random_state=42).fit_resample(X_train, y_train)


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (label, ys) in zip(axes, [
    ('Original',       y_train),
    ('After SMOTE',    y_sm),
    ('After ADASYN',   y_ad),
    ('After SMOTE-Tomek', y_st),
]):
    counts = pd.Series(ys).value_counts().sort_index()
    ax.bar(['Rejected','Approved'], counts.values, color=['#FF5252','#4CAF50'])
    ax.set_title(label, fontweight='bold')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 2, str(v), ha='center')
plt.suptitle('Class Distribution: Resampling Methods', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('09_resampling_class_distribution.png', dpi=150)
plt.show()


In [ ]:
# Train all XGBoost variants
xgb_smote = XGBClassifier(**xgb_params)
xgb_smote.fit(X_sm, y_sm, eval_set=[(X_test, y_test)], verbose=False)

spw = (y_train == 0).sum() / (y_train == 1).sum()
xgb_w = XGBClassifier(**{**xgb_params, 'scale_pos_weight': spw})
xgb_w.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

xgb_ad = XGBClassifier(**xgb_params)
xgb_ad.fit(X_ad, y_ad, eval_set=[(X_test, y_test)], verbose=False)

xgb_st = XGBClassifier(**xgb_params)
xgb_st.fit(X_st, y_st, eval_set=[(X_test, y_test)], verbose=False)

for name, mdl in [('Baseline', xgb), ('SMOTE', xgb_smote),
                   ('Weights', xgb_w), ('ADASYN', xgb_ad), ('SMOTE-Tomek', xgb_st)]:
    evaluate(f"XGBoost + {name}", y_test, mdl.predict(X_test),
             mdl.predict_proba(X_test)[:,1])


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(24, 4))
xgb_variants = [
    ('Baseline',    xgb),
    ('SMOTE',       xgb_smote),
    ('Weights',     xgb_w),
    ('ADASYN',      xgb_ad),
    ('SMOTE-Tomek', xgb_st),
]
for ax, (name, mdl) in zip(axes, xgb_variants):
    plot_confusion_matrix(ax, y_test, mdl.predict(X_test), f'XGBoost\n{name}')
plt.suptitle('XGBoost — Confusion Matrices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('10_xgboost_confusion_matrices.png', dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(9, 6))
for name, mdl in xgb_variants:
    proba = mdl.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
plt.plot([0,1],[0,1],'k--', linewidth=0.8)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('XGBoost — ROC Curves', fontweight='bold')
plt.legend(); plt.grid(); plt.tight_layout()
plt.savefig('11_xgboost_roc_curves.png', dpi=150)
plt.show()


In [ ]:
proba_smote = xgb_smote.predict_proba(X_test)[:,1]
thresholds = np.arange(0.3, 0.7, 0.05)
recall_0s, recall_1s = [], []

for t in thresholds:
    preds = (proba_smote >= t).astype(int)
    r = classification_report(y_test, preds, output_dict=True)
    recall_0s.append(r['0']['recall'])
    recall_1s.append(r['1']['recall'])

plt.figure(figsize=(9, 5))
plt.plot(thresholds, recall_0s, marker='o', label='Recall — Rejected (0)', color='#FF5252')
plt.plot(thresholds, recall_1s, marker='s', label='Recall — Approved (1)', color='#4CAF50')
plt.xlabel('Decision Threshold'); plt.ylabel('Recall')
plt.title('XGBoost + SMOTE — Threshold Tuning', fontweight='bold')
plt.legend(); plt.grid(); plt.tight_layout()
plt.savefig('12_xgboost_threshold_tuning.png', dpi=150)
plt.show()


In [ ]:
importances = pd.Series(xgb_smote.feature_importances_, index=X_train.columns)
importances_sorted = importances.sort_values(ascending=True)

plt.figure(figsize=(10, 7))
plt.barh(importances_sorted.index, importances_sorted.values, color='steelblue')
plt.xlabel('Feature Importance Score')
plt.title('XGBoost Feature Importances', fontweight='bold')
plt.tight_layout()
plt.savefig('13_xgboost_feature_importance.png', dpi=150)
plt.show()


## 3. Decision Tree Classifier

In [ ]:
dt_params = dict(criterion='entropy', max_depth=5,
                 min_samples_split=10, min_samples_leaf=5, random_state=42)

dt          = DecisionTreeClassifier(**dt_params)
dt_weighted = DecisionTreeClassifier(**{**dt_params, 'class_weight': 'balanced'})

dt.fit(X_train, y_train)
dt_weighted.fit(X_train, y_train)

evaluate("Decision Tree Baseline",        y_test, dt.predict(X_test))
evaluate("Decision Tree + Balanced Weights", y_test, dt_weighted.predict(X_test),
         dt_weighted.predict_proba(X_test)[:,1])


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_confusion_matrix(axes[0], y_test, dt.predict(X_test), 'DT Baseline')
plot_confusion_matrix(axes[1], y_test, dt_weighted.predict(X_test), 'DT + Balanced Weights')

# ROC for DT weighted
fpr, tpr, _ = roc_curve(y_test, dt_weighted.predict_proba(X_test)[:,1])
auc = roc_auc_score(y_test, dt_weighted.predict_proba(X_test)[:,1])
axes[2].plot(fpr, tpr, color='steelblue', label=f'DT Weighted (AUC={auc:.3f})')
axes[2].plot([0,1],[0,1],'k--', linewidth=0.8)
axes[2].set_xlabel('FPR'); axes[2].set_ylabel('TPR')
axes[2].set_title('Decision Tree ROC Curve', fontweight='bold')
axes[2].legend(); axes[2].grid()

plt.suptitle('Decision Tree Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('14_decision_tree_results.png', dpi=150)
plt.show()


In [ ]:
dt_importances = pd.Series(dt_weighted.feature_importances_, index=X_train.columns)
dt_importances_sorted = dt_importances[dt_importances > 0].sort_values(ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(dt_importances_sorted.index, dt_importances_sorted.values, color='darkorange')
plt.xlabel('Feature Importance Score')
plt.title('Decision Tree — Feature Importances', fontweight='bold')
plt.tight_layout()
plt.savefig('15_dt_feature_importance.png', dpi=150)
plt.show()


## 4. K-Nearest Neighbours

In [ ]:
scaler_knn = StandardScaler()
X_tr_knn = scaler_knn.fit_transform(X_train)
X_te_knn  = scaler_knn.transform(X_test)


In [ ]:
k_range = range(1, 40)
accs = [accuracy_score(y_test,
        KNeighborsClassifier(n_neighbors=k).fit(X_tr_knn, y_train).predict(X_te_knn))
        for k in k_range]

plt.figure(figsize=(10, 4))
plt.plot(k_range, accs, marker='o', color='steelblue')
plt.xlabel('K'); plt.ylabel('Accuracy')
plt.title('KNN — Accuracy vs K', fontweight='bold')
plt.grid(); plt.tight_layout()
plt.savefig('16_knn_accuracy_vs_k.png', dpi=150)
plt.show()


In [ ]:
# Grid search best K
grid = GridSearchCV(KNeighborsClassifier(),
                    {'n_neighbors': list(range(1,40)),
                     'metric': ['euclidean','manhattan']},
                    cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_tr_knn, y_train)
best_knn = grid.best_estimator_
print("Best params:", grid.best_params_)
evaluate("KNN Best", y_test, best_knn.predict(X_te_knn),
         best_knn.predict_proba(X_te_knn)[:,1])


In [ ]:
# KNN + SMOTE
X_sm_knn, y_sm_knn = SMOTE(random_state=42).fit_resample(X_tr_knn, y_train)
grid_sm = GridSearchCV(KNeighborsClassifier(),
                       {'n_neighbors': list(range(3,30,2)),
                        'metric': ['euclidean','manhattan','minkowski']},
                       cv=5, scoring='recall_macro', n_jobs=-1)
grid_sm.fit(X_sm_knn, y_sm_knn)
knn_smote = grid_sm.best_estimator_
print("Best SMOTE params:", grid_sm.best_params_)
evaluate("KNN + SMOTE", y_test, knn_smote.predict(X_te_knn),
         knn_smote.predict_proba(X_te_knn)[:,1])

# KNN + Distance Weighting
knn_dist = KNeighborsClassifier(n_neighbors=15, metric='manhattan', weights='distance')
knn_dist.fit(X_tr_knn, y_train)
evaluate("KNN + Distance Weights", y_test, knn_dist.predict(X_te_knn),
         knn_dist.predict_proba(X_te_knn)[:,1])


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
knn_variants = [('Best K', best_knn, X_te_knn),
                ('SMOTE',  knn_smote, X_te_knn),
                ('Dist Weights', knn_dist, X_te_knn)]
for ax, (name, mdl, Xte) in zip(axes, knn_variants):
    plot_confusion_matrix(ax, y_test, mdl.predict(Xte), f'KNN — {name}')
plt.suptitle('KNN — Confusion Matrices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('17_knn_confusion_matrices.png', dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(9, 6))
for name, mdl, Xte in knn_variants:
    proba = mdl.predict_proba(Xte)[:,1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
plt.plot([0,1],[0,1],'k--', linewidth=0.8)
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('KNN — ROC Curves', fontweight='bold')
plt.legend(); plt.grid(); plt.tight_layout()
plt.savefig('18_knn_roc_curves.png', dpi=150)
plt.show()


## 5. Naive Bayes Variants

In [ ]:
# Gaussian NB
gnb = GaussianNB()
gnb.fit(X_train, y_train)
evaluate("Gaussian NB", y_test, gnb.predict(X_test), gnb.predict_proba(X_test)[:,1])

# Gaussian NB + SMOTE
X_sm2, y_sm2 = SMOTE(random_state=42).fit_resample(X_train, y_train)
gnb_sm = GaussianNB()
gnb_sm.fit(X_sm2, y_sm2)
evaluate("Gaussian NB + SMOTE", y_test, gnb_sm.predict(X_test),
         gnb_sm.predict_proba(X_test)[:,1])

# Gaussian NB + Class Weights (sample_weight)
class_counts = y_train.value_counts()
total = len(y_train)
weights = np.where(y_train == 1,
                   total / (2 * class_counts[1]),
                   total / (2 * class_counts[0]))
gnb_w = GaussianNB()
gnb_w.fit(X_train, y_train, sample_weight=weights)
evaluate("Gaussian NB + Class Weights", y_test, gnb_w.predict(X_test),
         gnb_w.predict_proba(X_test)[:,1])


In [ ]:
# Multinomial NB (non-negative features via MinMaxScaler)
X_tr_mm = MinMaxScaler().fit_transform(X_train)
X_te_mm = MinMaxScaler().fit_transform(X_test)
mnb = MultinomialNB()
mnb.fit(X_tr_mm, y_train)
evaluate("Multinomial NB", y_test, mnb.predict(X_te_mm))

# Bernoulli NB (binarised features)
thresh = X_train.mean().mean()
X_tr_bin = Binarizer(threshold=thresh).fit_transform(X_train)
X_te_bin = Binarizer(threshold=thresh).transform(X_test)
bnb = BernoulliNB()
bnb.fit(X_tr_bin, y_train)
evaluate("Bernoulli NB", y_test, bnb.predict(X_te_bin))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
nb_proba_variants = [('Gaussian NB', gnb, X_test),
                     ('GNB + SMOTE', gnb_sm, X_test),
                     ('GNB + Weights', gnb_w, X_test)]
for ax, (name, mdl, Xte) in zip(axes, nb_proba_variants):
    plot_confusion_matrix(ax, y_test, mdl.predict(Xte), f'NB — {name}')
plt.suptitle('Naive Bayes — Confusion Matrices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('19_nb_confusion_matrices.png', dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
for name, mdl, Xte in nb_proba_variants:
    probs = mdl.predict_proba(Xte)[:,1]
    thresholds = np.arange(0, 1.01, 0.01)
    accs = [accuracy_score(y_test, (probs>=t).astype(int)) for t in thresholds]
    plt.plot(thresholds, accs, label=name)
plt.xlabel('Threshold'); plt.ylabel('Accuracy')
plt.title('Naive Bayes — Accuracy vs Decision Threshold', fontweight='bold')
plt.legend(); plt.grid(); plt.tight_layout()
plt.savefig('20_nb_threshold_curves.png', dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(9, 6))
for name, mdl, Xte in nb_proba_variants:
    proba = mdl.predict_proba(Xte)[:,1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
plt.plot([0,1],[0,1],'k--', linewidth=0.8)
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('Naive Bayes — ROC Curves', fontweight='bold')
plt.legend(); plt.grid(); plt.tight_layout()
plt.savefig('21_nb_roc_curves.png', dpi=150)
plt.show()


## 6. Model Comparison Summary

In [ ]:
results = {
    'XGBoost Baseline':       accuracy_score(y_test, xgb.predict(X_test)),
    'XGBoost + SMOTE':        accuracy_score(y_test, xgb_smote.predict(X_test)),
    'XGBoost + Weights':      accuracy_score(y_test, xgb_w.predict(X_test)),
    'XGBoost + ADASYN':       accuracy_score(y_test, xgb_ad.predict(X_test)),
    'XGBoost + SMOTE-Tomek':  accuracy_score(y_test, xgb_st.predict(X_test)),
    'DT Baseline':            accuracy_score(y_test, dt.predict(X_test)),
    'DT + Balanced Weights':  accuracy_score(y_test, dt_weighted.predict(X_test)),
    'KNN Best':               accuracy_score(y_test, best_knn.predict(X_te_knn)),
    'KNN + SMOTE':            accuracy_score(y_test, knn_smote.predict(X_te_knn)),
    'KNN + Dist Weights':     accuracy_score(y_test, knn_dist.predict(X_te_knn)),
    'Gaussian NB':            accuracy_score(y_test, gnb.predict(X_test)),
    'GNB + SMOTE':            accuracy_score(y_test, gnb_sm.predict(X_test)),
    'GNB + Weights':          accuracy_score(y_test, gnb_w.predict(X_test)),
    'Multinomial NB':         accuracy_score(y_test, mnb.predict(X_te_mm)),
    'Bernoulli NB':           accuracy_score(y_test, bnb.predict(X_te_bin)),
}

res_df = pd.DataFrame.from_dict(results, orient='index', columns=['Accuracy'])
res_df.sort_values('Accuracy', ascending=True, inplace=True)

plt.figure(figsize=(12, 8))
colors = ['#4CAF50' if v == res_df['Accuracy'].max() else 'steelblue'
          for v in res_df['Accuracy']]
plt.barh(res_df.index, res_df['Accuracy'], color=colors)
plt.xlabel('Accuracy'); plt.title('All Models — Accuracy Comparison', fontweight='bold')
plt.xlim(0.55, 1.0)
for i, v in enumerate(res_df['Accuracy']):
    plt.text(v + 0.003, i, f'{v:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('22_model_comparison.png', dpi=150)
plt.show()

print("\n🏆 Best model:", res_df['Accuracy'].idxmax(),
      f"→ {res_df['Accuracy'].max():.4f}")


## 7. Save Best Model
> **Decision Tree with Balanced Weights** selected — best overall recall for both classes, important in risk-based lending.

In [ ]:
joblib.dump(dt_weighted, 'dt_weighted_model.pkl')
joblib.dump(list(X_train.columns), 'training_columns.pkl')
print("✅ Saved: dt_weighted_model.pkl  |  training_columns.pkl")


## 8. Predict New Customers

In [ ]:
caps       = joblib.load('log_caps.pkl')
scaler     = joblib.load('scaler.pkl')
rf_reg     = joblib.load('loanamount_regressor.pkl')
ohe        = joblib.load('loanamount_ohe.pkl')
train_cols = joblib.load('training_columns.pkl')
model      = joblib.load('dt_weighted_model.pkl')

new_df = pd.read_csv('Datasets/New Customer.csv')
new_df.drop('Loan_ID', axis=1, errors='ignore', inplace=True)
print("New customers:", new_df.shape)
new_df.head()


In [ ]:
# ── Replicate Phase 1 preprocessing ─────────────────────────

# 1. Simple imputation
new_df['Gender'].fillna(new_df['Gender'].mode()[0], inplace=True)
new_df.loc[new_df['Married'].isnull() & (new_df['Education']=='Graduate'),     'Married'] = 'Yes'
new_df.loc[new_df['Married'].isnull() & (new_df['Education']=='Not Graduate'), 'Married'] = 'No'
new_df['Dependents']      = new_df.groupby('Married')['Dependents'].transform(
    lambda x: x.fillna(x.mode()[0]))
new_df['Self_Employed'].fillna(new_df['Self_Employed'].mode()[0], inplace=True)
new_df['Loan_Amount_Term'].fillna(360.0, inplace=True)
new_df['Credit_History'].fillna(1.0, inplace=True)

# 2. LoanAmount RF imputation
rf_features = ['ApplicantIncome','CoapplicantIncome','Loan_Amount_Term',
               'Credit_History','Gender','Married','Dependents',
               'Education','Self_Employed','Property_Area']
cat_cols = ohe.feature_names_in_
missing_mask = new_df['LoanAmount'].isnull()
if missing_mask.any():
    X_miss = new_df.loc[missing_mask, rf_features]
    X_cat  = pd.DataFrame(ohe.transform(X_miss[cat_cols]),
                          columns=ohe.get_feature_names_out(cat_cols))
    X_num  = X_miss.drop(columns=cat_cols).reset_index(drop=True)
    new_df.loc[missing_mask, 'LoanAmount'] = rf_reg.predict(
        pd.concat([X_num, X_cat], axis=1))

# 3. Feature engineering + log + cap
new_df['TotalIncome']     = new_df['ApplicantIncome'] + new_df['CoapplicantIncome']
new_df['LoanAmount_log']  = np.clip(np.log(new_df['LoanAmount'] + 1),
                                    None, caps['LoanAmount_log'])
new_df['TotalIncome_log'] = np.clip(np.log(new_df['TotalIncome'] + 1),
                                    None, caps['TotalIncome_log'])

# 4. Encoding
new_df['Dependents'] = new_df['Dependents'].replace('3+', 3).astype(int)
new_df['Education']  = new_df['Education'].map({'Not Graduate': 0, 'Graduate': 1})
new_df = pd.get_dummies(new_df,
                        columns=['Gender','Married','Self_Employed','Property_Area'],
                        drop_first=True)

# 5. Scale
new_df[['LoanAmount_log','TotalIncome_log','Loan_Amount_Term']] = scaler.transform(
    new_df[['LoanAmount_log','TotalIncome_log','Loan_Amount_Term']])

# 6. Align columns
X_new = new_df.reindex(columns=train_cols, fill_value=0)
print("Preprocessed new customers:", X_new.shape)


In [ ]:
y_pred = model.predict(X_new)
new_df['Loan_Status_Pred'] = y_pred

results_out = pd.DataFrame({
    'Loan_Status_Pred':  y_pred,
    'Loan_Status_Label': pd.Series(y_pred).map({1: 'Approved', 0: 'Rejected'}).values
}, index=new_df.index)

results_out.to_csv('Outputs/loan_status_predictions.csv', index=True)
print("✅ Predictions saved.")
print(results_out['Loan_Status_Label'].value_counts())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

pred_counts = results_out['Loan_Status_Label'].value_counts()
axes[0].bar(pred_counts.index, pred_counts.values,
            color=['#4CAF50','#FF5252'])
axes[0].set_title('Overall Loan Predictions', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(pred_counts.values):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

axes[1].pie(pred_counts.values, labels=pred_counts.index,
            autopct='%1.1f%%', colors=['#4CAF50','#FF5252'], startangle=90)
axes[1].set_title('Prediction Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig('23_prediction_overview.png', dpi=150)
plt.show()


In [ ]:
married_col    = 'Married_Yes'         if 'Married_Yes'         in new_df.columns else None
semiurban_col  = 'Property_Area_Semiurban' if 'Property_Area_Semiurban' in new_df.columns else None
urban_col      = 'Property_Area_Urban' if 'Property_Area_Urban' in new_df.columns else None

if married_col and semiurban_col:
    subset = new_df[(new_df[married_col] == True) & (new_df[semiurban_col] == True)]
    approved = subset['Loan_Status_Pred'].sum()
    total    = len(subset)
    pct      = (approved / total * 100) if total > 0 else 0
    print(f"Married + Semiurban applicants: {total}")
    print(f"  Approved: {approved} ({pct:.1f}%)")
    print(f"  Rejected: {total - approved} ({100-pct:.1f}%)")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # countplot
    subset_copy = subset.copy()
    subset_copy['Status'] = subset_copy['Loan_Status_Pred'].map({1:'Approved',0:'Rejected'})
    subset_copy['Status'].value_counts().plot.bar(
        ax=axes[0], color=['#4CAF50','#FF5252'], rot=0)
    axes[0].set_title('Married + Semiurban\nLoan Outcomes', fontweight='bold')
    axes[0].set_ylabel('Count')

    # pie chart
    axes[1].pie([approved, total - approved],
                labels=['Approved','Rejected'],
                autopct='%1.1f%%', colors=['#4CAF50','#FF5252'], startangle=90)
    axes[1].set_title('Married + Semiurban\nApproval Distribution', fontweight='bold')

    plt.tight_layout()
    plt.savefig('24_married_semiurban_analysis.png', dpi=150)
    plt.show()
else:
    print("Married_Yes / Property_Area_Semiurban columns not found in new_df.")


In [ ]:
if urban_col:
    fig, ax = plt.subplots(figsize=(8, 5))
    prop_area_labels = {0: 'Semiurban', 1: 'Urban'}
    plot_df = new_df[[urban_col, 'Loan_Status_Pred']].copy()
    plot_df['Area']   = plot_df[urban_col].map(prop_area_labels)
    plot_df['Status'] = plot_df['Loan_Status_Pred'].map({1:'Approved',0:'Rejected'})

    sns.countplot(x='Area', hue='Status', data=plot_df,
                  palette={'Approved':'#4CAF50','Rejected':'#FF5252'}, ax=ax)
    ax.set_title('Loan Approval by Property Area\n(Urban vs Semiurban)', fontweight='bold')
    ax.set_xlabel('Property Area'); ax.set_ylabel('Count')
    plt.tight_layout()
    plt.savefig('25_loan_approval_by_property_area.png', dpi=150)
    plt.show()
